# Task 14 — Dự báo Nhu cầu 7 Ngày bằng Mạng Học Sâu `DLinear` (Bảng 3)

Notebook này đóng gói quy trình huấn luyện, kiểm chứng đầu vào SHA-256, đối soát log thực thi GPU và tổng hợp 9 dòng chỉ số chuẩn của cột **DLinear** trong Bảng 3 bài báo *FreshRetailNet-50K*:
1. **`DLinear_RawSales`** (`--data censored_dataset --target sale_amount --des raw_l1`)
2. **`DLinear_TimesNet`** (`--data recovered_dataset --target sale_amount_pred --des timesnet_l1`)
3. **`DLinear_DLinear`** (`--data recovered_dataset --target sale_amount_pred --des dlinear_l1`)

- **Mã gốc đối chiếu:** [`Dingdong-Inc/frn-50k-baseline` @ `8af694a`](https://github.com/Dingdong-Inc/frn-50k-baseline/tree/8af694a09a5a427d1d50648d348c022565a781f2/demand_forecasting/DLinear) (`fix_seed = 2025`, `revin=True` qua cờ gốc `--revin False`, `seq_len=62`, `moving_avg=28`, `pred_len=7`, `enc_in=7`, `batch_size=1024`, `train_epochs=6`, `lr=0.001`).
- **Môi trường đã huấn luyện lượt chính:** Windows CUDA GPU (`PyTorch 2.0.1+cu118`, `Python 3.8.20`, conda env `frn50k`). Đặt `TASK14_RUN_GPU=1` nếu muốn chạy huấn luyện lại từ đầu trên Colab GPU T4 hoặc máy CUDA.

In [1]:
from pathlib import Path
from hashlib import sha256
import json
import os
import re
import subprocess
import numpy as np
import pandas as pd

RUN_GPU_TRAINING = bool(int(os.environ.get('TASK14_RUN_GPU', '0')))

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')
    project_dir = Path('/content/PTDLKD')
    if not project_dir.is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/dangkhoauit184/PTDLKD.git', str(project_dir)], check=True)
else:
    cwd = Path.cwd().resolve()
    project_dir = next((p for p in (cwd, *cwd.parents) if (p / 'Task14_DLinear_Forecasting').is_dir()), cwd)

task_dir = project_dir / 'Task14_DLinear_Forecasting'
results_dir = task_dir / 'results'
train_path = project_dir / 'Task06_Data_Preparation/train.parquet'
eval_path  = project_dir / 'Task06_Data_Preparation/eval.parquet'

def resolve_recovered(fname: str, drive_id: str) -> Path:
    for cand in (task_dir / fname, project_dir / fname, project_dir / 'tmp' / fname, Path('/content/data') / fname):
        if cand.is_file():
            return cand
    dst = Path('/content/data' if drive is not None else project_dir / 'tmp') / fname
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['gdown', drive_id, '-O', str(dst)], check=True)
    return dst

timesnet_path = resolve_recovered('demand_timesnet.parquet', '1f9db-U9f9C9OFS-V2EcmQv4FIU3bguR3')
dlinear_path  = resolve_recovered('demand_dlinear.parquet',  '1MDcD7cOyzcnoBu48I6eGUkSfwx8b3hkh')

print(f'Project root     : {project_dir}')
print(f'RUN_GPU_TRAINING : {RUN_GPU_TRAINING}')

Project root     : D:\PTDLKD
RUN_GPU_TRAINING : False


## 1. Xác nhận SHA-256 của 4 tệp Parquet đầu vào và quy tắc phân nhóm `psd` gốc

Đảm bảo cả 4 tệp Parquet khớp tuyệt đối mã SHA-256 của Task 6, Task 10 và Task 12; xác nhận `psd` tính từ `sale_amount` gốc của `train.parquet` chia đúng `206.562` dòng `eval` không hết hàng thành `Low Sale = 153.208` và `High Sale = 53.354` (trùng khớp 100% với Task 8, 11, 13).

In [2]:
EXPECTED_SHA256 = {
    'train.parquet':           ('b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96', train_path,    4_500_000),
    'eval.parquet':            ('61cfd3cb60d767d38ba9e640be64e982a15b2918ca0ba14fe0283c5a2eac6872', eval_path,       350_000),
    'demand_timesnet.parquet': ('c34f4eafe087d225865a624ef24d213af566496541458c3075f6e596d9f1d694', timesnet_path, 4_500_000),
    'demand_dlinear.parquet':  ('2cd9091adec89b129d0c726e3be658b9becf027b4ece1e3b725feb74176ab65b', dlinear_path,  4_500_000),
}

def sha256_file(path: Path) -> str:
    h = sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

for label, (exp_sha, path, exp_rows) in EXPECTED_SHA256.items():
    obs_sha = sha256_file(path)
    assert obs_sha == exp_sha, f'SHA-256 lệch cho {label}: {obs_sha}'
    n_rows = len(pd.read_parquet(path, columns=['store_id'], engine='pyarrow'))
    assert n_rows == exp_rows, f'Số dòng lệch cho {label}: {n_rows}'
    print(f'OK: {label:25s} | {path.stat().st_size:>10,} bytes | {n_rows:>9,} dòng | sha256={obs_sha[:16]}...')

df_tr_psd = pd.read_parquet(train_path, columns=['store_id', 'product_id', 'sale_amount'], engine='pyarrow')
psd_df = df_tr_psd.groupby(['store_id', 'product_id'], as_index=False)['sale_amount'].mean().rename(columns={'sale_amount': 'psd'})
df_ev = pd.read_parquet(eval_path, columns=['store_id', 'product_id', 'stock_hour6_22_cnt'], engine='pyarrow').merge(psd_df, on=['store_id', 'product_id'], how='left')
elig = df_ev.query('stock_hour6_22_cnt == 0')
n_all, n_low, n_high = len(elig), len(elig.query('psd < 1')), len(elig.query('psd >= 1'))
print(f'Phân nhóm eval hợp lệ (stock_hour6_22_cnt == 0): Overall = {n_all:,} | Low Sale = {n_low:,} | High Sale = {n_high:,}')
assert (n_all, n_low, n_high) == (206562, 153208, 53354)
del df_tr_psd, psd_df, df_ev, elig

OK: train.parquet             | 61,075,680 bytes | 4,500,000 dòng | sha256=b7e01c711bc4fa05...
OK: eval.parquet              |  5,451,872 bytes |   350,000 dòng | sha256=61cfd3cb60d767d3...
OK: demand_timesnet.parquet   | 81,562,029 bytes | 4,500,000 dòng | sha256=c34f4eafe087d225...
OK: demand_dlinear.parquet    | 81,531,473 bytes | 4,500,000 dòng | sha256=2cd9091adec89b12...
Phân nhóm eval hợp lệ (stock_hour6_22_cnt == 0): Overall = 206,562 | Low Sale = 153,208 | High Sale = 53,354


## 2. Kiểm tra bản vá `code_changes.diff` và cấu hình huấn luyện GPU

Bản vá tối thiểu trên `demand_forecasting/DLinear` (commit `8af694a`) chỉ bổ sung `--train_path` và `--eval_path` để `Exp_Main.data_organization` đọc trực tiếp Parquet local thay vì gọi `load_dataset("Dingdong-Inc/FreshRetailNet-50K")` (nhờ đó `psd` luôn được tính từ `sale_amount` gốc của `train.parquet`), đồng thời giữ nguyên toàn bộ kiến trúc `DLinear`, `RevIN` (`--revin False` → `bool("False") == True`), `seed=2025` và bộ siêu tham số của tác giả.

In [3]:
diff_file = task_dir / 'code_changes.diff'
assert diff_file.is_file()
baseline_dir = project_dir / 'frn-50k-baseline'
if (baseline_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(baseline_dir), 'apply', '--check', str(diff_file)], check=True)
    print('Kiểm tra git apply --check trên frn-50k-baseline @ 8af694a: PASS')

COMMON_FLAGS = [
    '--is_training', '1', '--train_only', 'True', '--model', 'DLinear',
    '--scale', 'True', '--loss', 'mae', '--features', 'MS',
    '--revin', 'False', '--total_seq_len', '90', '--seq_len', '62',
    '--pred_len', '7', '--moving_avg', '28', '--enc_in', '7',
    '--do_predict', '--itr', '1', '--batch_size', '1024',
    '--train_epochs', '6', '--learning_rate', '0.001',
]
CONFIG_SPECS = [
    ('DLinear_RawSales', 'censored_dataset',  train_path,    'sale_amount',      'raw_l1',      'dlinear_raw_sales.log'),
    ('DLinear_TimesNet', 'recovered_dataset', timesnet_path, 'sale_amount_pred', 'timesnet_l1', 'dlinear_timesnet.log'),
    ('DLinear_DLinear',  'recovered_dataset', dlinear_path,  'sale_amount_pred', 'dlinear_l1',  'dlinear_dlinear.log'),
]
for cid, dtype, dpath, tgt, des, lfile in CONFIG_SPECS:
    print(f'{cid:18s} -> --data {dtype:18s} --target {tgt:16s} --des {des:12s} (log: {lfile})')

if RUN_GPU_TRAINING:
    work_repo = Path('/content/frn-50k-baseline') if drive is not None else baseline_dir
    if not (work_repo / '.git').is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/Dingdong-Inc/frn-50k-baseline.git', str(work_repo)], check=True)
    subprocess.run(['git', '-C', str(work_repo), 'checkout', '--detach', '8af694a09a5a427d1d50648d348c022565a781f2'], check=True)
    subprocess.run(['git', '-C', str(work_repo), 'checkout', '--', '.'], check=True)
    subprocess.run(['git', '-C', str(work_repo), 'apply', str(diff_file)], check=True)
    dlinear_cwd = work_repo / 'demand_forecasting/DLinear'
    n_workers = '2' if os.name != 'nt' else '0'
    for cid, dtype, dpath, tgt, des, lfile in CONFIG_SPECS:
        cmd = ['python', '-u', 'main.py', *COMMON_FLAGS, '--num_workers', n_workers,
               '--data', dtype, '--data_path', str(dpath),
               '--train_path', str(train_path), '--eval_path', str(eval_path),
               '--target', tgt, '--des', des]
        with (results_dir / 'full' / lfile).open('w', encoding='utf-8') as lf:
            subprocess.run(cmd, cwd=dlinear_cwd, stdout=lf, stderr=subprocess.STDOUT, check=True)

Kiểm tra git apply --check trên frn-50k-baseline @ 8af694a: PASS
DLinear_RawSales   -> --data censored_dataset   --target sale_amount      --des raw_l1       (log: dlinear_raw_sales.log)
DLinear_TimesNet   -> --data recovered_dataset  --target sale_amount_pred --des timesnet_l1  (log: dlinear_timesnet.log)
DLinear_DLinear    -> --data recovered_dataset  --target sale_amount_pred --des dlinear_l1   (log: dlinear_dlinear.log)


## 3. Đối soát Log thực thi GPU, Bảng chỉ số Task 14 (9 dòng) và So sánh Cột `SSA` vs `DLinear`

Kiểm tra chéo từng tệp log trong `results/full/` (`train 1100000`, `6/6` epoch, `pred 50000`, `revin=True`, tổng mẫu mỗi ngày `21.886,857143 + 7.622,0 = 29.508,857143` × 7 ngày = `206.562` dòng, khớp `mae` tới 4 chữ số thập phân) và đặt cạnh kết quả 3 cấu hình cột SSA (Task 8, 11, 13).

In [4]:
df_metrics = pd.read_csv(results_dir / 'dlinear_forecasting_metrics.csv', keep_default_na=False)
meta = json.loads((results_dir / 'run_metadata.json').read_text(encoding='utf-8'))

smoke_txt = (results_dir / 'smoke/dlinear_smoke.log').read_text(encoding='utf-8')
assert 'train 4400' in smoke_txt and 'pred 200' in smoke_txt and 'revin=True' in smoke_txt
print('Smoke log (200 cặp): train=4,400 cửa sổ | pred=200 chuỗi | revin=True -> OK')

group_map = {'Overall': 'psd>=0', 'Low Sale': 'psd<1', 'High Sale': 'psd>=1'}
for cid, _, _, _, des, lfile in CONFIG_SPECS:
    txt = (results_dir / 'full' / lfile).read_text(encoding='utf-8')
    assert 'train 1100000' in txt and 'pred 50000' in txt and 'revin=True' in txt
    losses = re.findall(r'Epoch:\s*\d+,\s*Steps:\s*1074\s*\|\s*Train Loss:\s*([0-9.]+)', txt)
    assert len(losses) == 6, f'{cid}: chưa đủ 6 epoch!'
    # Kiểm tra sample_cnt * 7 == 206562 và khớp MAE với CSV
    m_all = re.search(r'psd>=0\s+([0-9.]+)\s+[0-9.]+\s+\.\.\.\s+([0-9.]+)\s+[-0-9.]+', txt)
    assert m_all and round(float(m_all.group(1)) * 7) == 206562
    for grp, gcode in group_map.items():
        m_row = re.search(rf'{re.escape(gcode)}\s+([0-9.]+)\s+[0-9.]+\s+\.\.\.\s+([0-9.]+)\s+([-0-9.]+)', txt)
        assert m_row, f'Thiếu dòng {gcode} trong {lfile}'
        log_mae = round(float(m_row.group(2)), 4)
        csv_mae = float(df_metrics.loc[(df_metrics.config_id == cid) & (df_metrics.group == grp), 'mae'].iloc[0])
        assert abs(log_mae - csv_mae) < 1e-4, f'Lệch MAE {cid} {grp}: {log_mae} vs {csv_mae}'
    print(f'{cid:18s}: 6/6 epochs (loss {losses[0]} -> {losses[-1]}) | eligible=206,562 | MAE khớp CSV 100%')

print('\n=== BẢNG CHỈ SỐ TASK 14: CỘT DLINEAR BẢNG 3 (9 DÒNG) ===')
print(df_metrics.to_string(index=False))

ssa_summary_path = project_dir / 'Task13_SSA_DLinear/results/ssa_all_3configs_summary.csv'
if ssa_summary_path.is_file():
    df_ssa = pd.read_csv(ssa_summary_path, keep_default_na=False)
    print('\n=== ĐỐI CHIẾU SONG SONG: CỘT SSA (TASK 8, 13, 11) vs CỘT DLINEAR (TASK 14) ===')
    comp_rows = []
    for grp in ('Overall', 'Low Sale', 'High Sale'):
        for src_label, ssa_id, dl_id in (
            ('Raw Sales', 'SSA_RawSales', 'DLinear_RawSales'),
            ('DLinear Recovery', 'SSA_DLinear', 'DLinear_DLinear'),
            ('TimesNet Recovery', 'SSA_TimesNet', 'DLinear_TimesNet'),
        ):
            r_s = df_ssa.loc[(df_ssa.config_id == ssa_id) & (df_ssa.group == grp)].iloc[0]
            r_d = df_metrics.loc[(df_metrics.config_id == dl_id) & (df_metrics.group == grp)].iloc[0]
            comp_rows.append({
                'group': grp,
                'demand_source': src_label,
                'SSA_WAPE': r_s['wape_pct'],
                'SSA_WPE': r_s['wpe_pct'],
                'DLinear_WAPE': r_d['wape_pct'],
                'DLinear_WPE': r_d['wpe_pct'],
            })
    print(pd.DataFrame(comp_rows).to_string(index=False))

Smoke log (200 cặp): train=4,400 cửa sổ | pred=200 chuỗi | revin=True -> OK
DLinear_RawSales  : 6/6 epochs (loss 0.2660257 -> 0.2624925) | eligible=206,562 | MAE khớp CSV 100%
DLinear_TimesNet  : 6/6 epochs (loss 0.2209927 -> 0.2181068) | eligible=206,562 | MAE khớp CSV 100%
DLinear_DLinear   : 6/6 epochs (loss 0.2449368 -> 0.2416589) | eligible=206,562 | MAE khớp CSV 100%

=== BẢNG CHỈ SỐ TASK 14: CỘT DLINEAR BẢNG 3 (9 DÒNG) ===
       config_id     group  wape_pct  wpe_pct    mae paper_wape_pct paper_wpe_pct delta_wape_pp delta_wpe_pp    status
DLinear_RawSales   Overall     31.54    -5.21 0.3932          31.56         -4.89         -0.02        -0.32 completed
DLinear_RawSales  Low Sale     39.71    -1.31 0.2781          39.78         -1.03         -0.07        -0.28 completed
DLinear_RawSales High Sale     25.70    -7.97 0.7310          25.68         -7.63          0.02        -0.34 completed
DLinear_TimesNet   Overall     30.55     6.21 0.3797          30.73          6.88         